In [1]:
import xgboost as xgb
print(xgb.__version__)

3.4.1


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Projects_Supervisedlearning\Regression_HousepricePrediction\.venv\Scripts\python.exe
3.4.1
c:\Projects_Supervisedlearning\Regression_HousepricePrediction\.venv\Lib\site-packages\xgboost\__init__.py


In [12]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

In [ ]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost


NameError: name 'os' is not defined

In [4]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv(r"C:\Projects_Supervisedlearning\Regression_HousepricePrediction\data\processed\feature_engineered_train.csv")
eval_df  = pd.read_csv(r"C:\Projects_Supervisedlearning\Regression_HousepricePrediction\data\processed\feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 37)
Eval shape: (148448, 37)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [13]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder

mlflow.set_tracking_uri(
    r"C:\Projects_Supervisedlearning\Regression_HousepricePrediction\mlruns"
)

mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/10/12 21:35:15 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-10-12 21:35:15,568] A new study created in memory with name: no-name-56c9c3d9-c578-43ed-8fa7-cf59dfa31579


[I 2026-10-12 21:35:45,940] Trial 0 finished with value: 68469.86221589115 and parameters: {'n_estimators': 450, 'max_depth': 8, 'learning_rate': 0.019271868196991775, 'subsample': 0.7657515726059267, 'colsample_bytree': 0.5915245449554141, 'min_child_weight': 5, 'gamma': 1.4664198937516082, 'reg_alpha': 0.06387786889027752, 'reg_lambda': 1.3537530032962597}. Best is trial 0 with value: 68469.86221589115.
[I 2026-10-12 21:36:24,966] Trial 1 finished with value: 67724.91532969428 and parameters: {'n_estimators': 712, 'max_depth': 9, 'learning_rate': 0.02409625588692734, 'subsample': 0.5272791887779147, 'colsample_bytree': 0.7385793466834819, 'min_child_weight': 1, 'gamma': 3.6161673489948787, 'reg_alpha': 0.8995066992241203, 'reg_lambda': 0.002891304762175457}. Best is trial 1 with value: 67724.91532969428.
[I 2026-10-12 21:36:46,406] Trial 2 finished with value: 72126.4697386651 and parameters: {'n_estimators': 331, 'max_depth': 9, 'learning_rate': 0.012225297916360336, 'subsample': 0.

Best params: {'n_estimators': 446, 'max_depth': 8, 'learning_rate': 0.02837475128939735, 'subsample': 0.5745752035146263, 'colsample_bytree': 0.681500214899905, 'min_child_weight': 1, 'gamma': 3.2724424502941925, 'reg_alpha': 0.0003828302051649605, 'reg_lambda': 1.4371845297218926}


In [14]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 31245.61282715225
RMSE: 66726.79415647355
R²: 0.9655919111408385
